In [17]:
# Import packages
import requests
import io
import pandas as pd
import seaborn as sbn
import matplotlib.pyplot as plt
from datetime import date, timedelta, datetime
from IPython.display import display, Markdown, HTML
import numpy as np

In [10]:
###################
#SET E-MAIL HEADER#
###################

# This cell will serve as a header in your email. You can add some information here that may be useful for providing context. 
# If you want to change the actual text that appears, feel free to edit the "md_text" variable directly.

# Input a title of your choosing here.
title = "Nearby Permits"

# Write a brief description about the analysis.
description = """
Permits pulled for work near Peter's House.
"""

# Get today's date
current_date = date.today()

# Print markdown header
md_text = f"""
## {title}
**Description**
{description}
**Data as of**  
{current_date}
"""

# Render it in the output
display(Markdown(md_text))


## Nearby Permits
**Description**

Permits pulled for work near Peter's House.

**Data as of**  
2026-09-15


In [5]:
###########
#PULL DATA#
###########

# Put the URL for your API request here. You can do this using the query builder in ArcGIS Online.
url = "https://services3.arcgis.com/dty2kHktVXHrqO8i/arcgis/rest/services/Building_Permits/FeatureServer/0/query"

# Type out a where clause here. 
# You can utilize an "f string" to make this filter dynamic.
# NOTE: For many feature layers, the maximum amount of records the ArcGIS Online API can query is 2,000. You'll need to perform multiple queries if you are reading in more than 2k records.
cutoff_date = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d')
where_clause = f"PROJECT_FILE_DATE >= DATE '{cutoff_date}'"
# example_field > example_value

# Set query parameters. Nothing here for you to do.
query_params = {
    "where": where_clause,       # The where clause from above.
    "returnGeometry": "false",    # We're not doing any work with spatial data. But if you want to make maps with your data, set to 'true'.
    "f": "json"                  # Tells the server to respond with JSON format.
}

## SUBMIT AND PARSE API REQUEST(S)
# Since in many cases we are limited to reading 2,000 records per query, this function will make several API requests to get all the records.
# It will also parse the request and extract the data into a list of records.

def get_data(url,query_params):
    """
    This function retrieves data from ArcGIS Online FeatureLayer via a series of GET requests. 
    It will pull data in groups of 2,000 records, and append all the data to one list of records.

    Args:
        url (str): Spark session context.
        query_params (dict): A spark DataFrame to geocode.

    Returns:
        list: The spark DataFrame, with new geocoded columns appended.
    """
    # Get record count
    record_count = requests.get(url,params={"where":query_params['where'],"returnCountOnly":"true","f":"json"}).json()['count']

    # Split record count into offsets
    offsets = range(0,record_count,2000)
    
    # List of data records
    results = []
    
    # Loop through offsets and get data for each offset
    for i,offset in enumerate(offsets):
        # Perform a GET request with the given offset
        query_params['resultOffset'] = offset
        req = requests.get(url,params=query_params)
        # Get JSON
        resp = req.json()

        # Extract data and append it to our final result
        data = [a['attributes'] for a in resp['features']]

        results += data
    
    # Ensure the number of records matches the record count of the Feature Layer.
    assert len(results) == record_count

    return results

data = get_data(url, query_params)
df = pd.DataFrame(data)


In [ ]:
def haversine_miles(lat1, lon1, lat2, lon2):
    R = 3958.8
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

target_lat, target_lon = 41.589689815437026, -81.55938540540967

df['DIST_MILES'] = haversine_miles(target_lat, target_lon, df['LAT'], df['LON'])

report_cols = [
    'PRIMARY_ADDRESS', 'FILE_DATE', 'ISSUE_DATE', 'JOB_DESCRIPTION',
    'WORK_DESCRIPTION', 'JOB_VALUE', 'CONTRACTOR_NAME', 'DIST_MILES'
]

nearby = df.loc[df['DIST_MILES'] <= 1.0, report_cols].sort_values('DIST_MILES').reset_index(drop=True)
nearby['FILE_DATE'] = pd.to_datetime(nearby['FILE_DATE'], unit='ms').dt.date
nearby['ISSUE_DATE'] = pd.to_datetime(nearby['ISSUE_DATE'], unit='ms').dt.date
nearby['DIST_MILES'] = nearby['DIST_MILES'].map(lambda x: f'{x:.2f}')

# Generate standard HTML and inject Gmail-safe inline styles
table_html = nearby.to_html(index=False)

styled_email_html = (
    table_html
    # Table layout
    .replace('<table', '<table style="border-collapse: collapse; width: 100%; font-family: -apple-system, BlinkMacSystemFont, \'Segoe UI\', Roboto, Arial, sans-serif; font-size: 14px; text-align: left; border: 1px solid #e2e8f0; box-shadow: 0 1px 3px rgba(0,0,0,0.05);"')
    .replace('<thead>', '<thead style="background-color: #f8fafc; color: #475569; font-size: 13px; text-transform: uppercase; letter-spacing: 0.5px;">')
    
    # ISOLATE WORK_DESCRIPTION: Temporarily rename it so the generic header styling doesn't overwrite it
    .replace('<th>WORK_DESCRIPTION</th>', '<th_wd>WORK_DESCRIPTION</th_wd>')
    
    # Generic header styling for all other columns
    .replace('<th', '<th style="padding: 14px 16px; font-weight: 600; border-bottom: 2px solid #cbd5e1; white-space: nowrap;"')
    
    # Apply specific styling to WORK_DESCRIPTION (forces it to be wider)
    .replace('<th_wd>', '<th style="padding: 14px 16px; font-weight: 600; border-bottom: 2px solid #cbd5e1; min-width: 300px; width: 35%;">')
    .replace('</th_wd>', '</th>')
    
    # Data rows
    .replace('<tr', '<tr style="background-color: #ffffff;"')
    .replace('<td', '<td style="padding: 14px 16px; border-bottom: 1px solid #e2e8f0; color: #334155; vertical-align: top; line-height: 1.4;"')
)

from IPython.display import HTML
HTML(styled_email_html)

PRIMARY_ADDRESS,FILE_DATE,ISSUE_DATE,JOB_DESCRIPTION,WORK_DESCRIPTION,JOB_VALUE,CONTRACTOR_NAME,DIST_MILES
"17401 TARRYMORE RD, CLEVELAND, OH, 44119",2026-08-24,2026-09-01,Interior/Exterior Remodel,MAKE INT/EXT ALT REPAIR/REPLACE 4 WINDOWS PER MANUF SPECS W/APPVD MATLS NO STRUCT CHANGES REMOVE DEBRIS,5911.0,DAVID FRIESORGER,0.14
"17836 CANTERBURY RD, CLEVELAND, OH, 44119",2026-09-01,2026-09-08,Add or change existing construction elements,MAKE INT/EXT ALT REPAIR/REPLACE 12 WINDOWS PER MANUF SPECS W/APPVD MATLS NO STRUCT CHANGES REMOVE DEBRIS,22366.0,WILLIAM A MARVIN,0.30
"18401 WINDWARD RD, CLEVELAND, OH, 44119",2026-08-26,2026-09-03,Interior Only Remodel,MAKE INT/EXT ALT REPAIR/REPLACE 8 WINDOWS PER MANUF SPECS W/APPVD MATLS NO STRUCT CHANGES REMOVE DEBRIS,6250.0,STEVE G COLOPY,0.43
"19101 MUSKOKA AVE, CLEVELAND, OH, 44119",2026-08-24,2026-09-11,Correct Violations,"MAKE INT/EXT ALT PER REHAB PLAN TO CORR. COND. V25036689 DATED 11/06/2025. OWNER HAS 48 HOURS TO SECURE PROPERTY, 60 DAYS TO CORR EXTERIOR VIOLATIONS AND 180 DAYS TO CORR ENTIRE NOTICE W/APP'D MATL'S PER CITY CODE. REMOVE ALL DEBRIS C OF O REQ'D, SEPARATE PERMITS REQ'D FOR M/E/P. FAILURE TO ADHERE TO THIS REHABILITATION PLAN COULD RESULT IN PROSECUTION OR DEMOLITION. THIS PERMIT MAY BE CLOSED IF WORK IS NOT COMPLETED IN 180 DAYS. COMPLETE ALL WORK REQ'D TO CORRECT VN FOR CERTIFICATE OF OCCUPANCY.",25000.0,PETER W MALONE,0.72
"19101 MUSKOKA AVE, CLEVELAND, OH, 44119",2026-09-11,2026-09-11,Residential Garage,RAZE GARAGE\r\nCOND V25037901,3800.0,PETER W MALONE,0.72
"18014 NOTTINGHAM RD, CLEVELAND, OH, 44119",2026-09-09,2026-09-10,Exterior only remodel,MAKE INT/EXT ALT REPAIR/REPLACE 1 DOOR PER MANUF SPECS W/APPVD MATLS NO STRUCT CHANGES REMOVE DEBRIS,8000.0,ANDREW J WEINFURTNER,0.85


In [0]:
######################
#CONVERT TO DATAFRAME#
######################
# If you use the get_data function from above, your data should look something like this:.
"""
[{'service_request_id': '202000403109',
  'service_category': 'Trash & Recycling',
  'service_name': 'Waste Cart Concerns'},
 {'service_request_id': '202000403083',
  'service_category': 'Building & Housing',
  'service_name': 'Electrical Issue'},
 {'service_request_id': '202000403082',
  'service_category': 'Street Issues',
  'service_name': 'Debris in Street'}]
"""

# The format above is known as "records" format, and will allow you to automatically convert to a pandas dataframe when doing pd.DataFrame(records).
# Try converting to DataFrame below:

In [12]:
##########
#ANALYSIS#
##########

# Conduct your data analysis below. You should use the dataframe from above as your starting point. Feel free to add more cells to separate output.
# We import the "seaborn" package in the first cell above. You can use this or another package of your choosing for creating visualizations.
# If you choose to import additional packages, be sure to update the dependencies in your GitHub Action! Otherwise the workflow will fail.